In [13]:
import time
import csv
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import resnet18, ResNet18_Weights

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_total_params(model):
    return sum(p.numel() for p in model.parameters())

def get_peak_memory():
    return torch.cuda.max_memory_allocated() / 1024**2  # MB

os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

csv_file = "results/ResNet18_Lora_metrics_april_21.csv"


if not os.path.exists(csv_file):
    with open(csv_file, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "method", "epoch",
            "train_loss", "train_acc",
            "val_loss", "val_acc",
            "epoch_time_sec",
            "peak_memory_mb",
            "trainable_params"
        ])


# Pretrained EfficientNetV2-S weights
weights = ResNet18_Weights.IMAGENET1K_V1



# Use simple preprocessing only
transform = weights.transforms()


# Load Food101
train_dataset_full = datasets.Food101(
    root="./data",
    split="train",
    download=True,
    transform=transform
)

test_dataset = datasets.Food101(
    root="./data",
    split="test",
    download=True,
    transform=transform
)

# Split training set into train + validation
train_size = int(0.9 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

Using device: cuda


In [14]:
import loralib as lora


def apply_lora_to_resnet(model, rank=8, alpha=16):
    def to_int(x):
        return x if isinstance(x, int) else x[0]

    def get_module(model, name):
        parts = name.split(".")
        m = model
        for p in parts:
            m = m[int(p)] if p.isdigit() else getattr(m, p)
        return m

    def set_module(model, name, new_module):
        parts = name.split(".")
        parent = model
        for p in parts[:-1]:
            parent = parent[int(p)] if p.isdigit() else getattr(parent, p)

        last = parts[-1]
        if last.isdigit():
            parent[int(last)] = new_module
        else:
            setattr(parent, last, new_module)

    target_layers = [
        "layer1.0.conv1", "layer1.0.conv2",
        "layer1.1.conv1", "layer1.1.conv2",
        "layer2.0.conv1", "layer2.0.conv2",
        "layer2.1.conv1", "layer2.1.conv2",
        "layer3.0.conv1", "layer3.0.conv2",
        "layer3.1.conv1", "layer3.1.conv2",
        "layer4.0.conv1", "layer4.0.conv2",
        "layer4.1.conv1", "layer4.1.conv2",
    ]

    for name in target_layers:
        module = get_module(model, name)

        new_layer = lora.Conv2d(
            in_channels=module.in_channels,
            out_channels=module.out_channels,
            kernel_size=to_int(module.kernel_size),
            stride=to_int(module.stride),
            padding=to_int(module.padding),
            dilation=to_int(module.dilation),
            groups=module.groups,
            bias=(module.bias is not None),
            r=rank,
            lora_alpha=alpha,
        )

        new_layer.conv.weight.data.copy_(module.weight.data)
        if module.bias is not None:
            new_layer.conv.bias.data.copy_(module.bias.data)

        set_module(model, name, new_layer)

    return model


In [15]:
model = models.resnet18(weights=weights)
model = apply_lora_to_resnet(model, rank=8, alpha=16)

model.fc = nn.Linear(model.fc.in_features, 101)

lora.mark_only_lora_as_trainable(model)

# Unfreeze classifier
for param in model.fc.parameters():
    param.requires_grad = True
model = model.to(device)


In [16]:
for name, p in model.named_parameters():
    if "lora" in name:
        print(name, p.shape)

layer1.0.conv1.lora_A torch.Size([24, 192])
layer1.0.conv1.lora_B torch.Size([192, 24])
layer1.0.conv2.lora_A torch.Size([24, 192])
layer1.0.conv2.lora_B torch.Size([192, 24])
layer1.1.conv1.lora_A torch.Size([24, 192])
layer1.1.conv1.lora_B torch.Size([192, 24])
layer1.1.conv2.lora_A torch.Size([24, 192])
layer1.1.conv2.lora_B torch.Size([192, 24])
layer2.0.conv1.lora_A torch.Size([24, 192])
layer2.0.conv1.lora_B torch.Size([384, 24])
layer2.0.conv2.lora_A torch.Size([24, 384])
layer2.0.conv2.lora_B torch.Size([384, 24])
layer2.1.conv1.lora_A torch.Size([24, 384])
layer2.1.conv1.lora_B torch.Size([384, 24])
layer2.1.conv2.lora_A torch.Size([24, 384])
layer2.1.conv2.lora_B torch.Size([384, 24])
layer3.0.conv1.lora_A torch.Size([24, 384])
layer3.0.conv1.lora_B torch.Size([768, 24])
layer3.0.conv2.lora_A torch.Size([24, 768])
layer3.0.conv2.lora_B torch.Size([768, 24])
layer3.1.conv1.lora_A torch.Size([24, 768])
layer3.1.conv1.lora_B torch.Size([768, 24])
layer3.1.conv2.lora_A torch.Size

In [17]:
for name, module in model.named_modules():
    if hasattr(module, "lora_A"):
        print("LoRA found in:", name)

LoRA found in: layer1.0.conv1
LoRA found in: layer1.0.conv2
LoRA found in: layer1.1.conv1
LoRA found in: layer1.1.conv2
LoRA found in: layer2.0.conv1
LoRA found in: layer2.0.conv2
LoRA found in: layer2.1.conv1
LoRA found in: layer2.1.conv2
LoRA found in: layer3.0.conv1
LoRA found in: layer3.0.conv2
LoRA found in: layer3.1.conv1
LoRA found in: layer3.1.conv2
LoRA found in: layer4.0.conv1
LoRA found in: layer4.0.conv2
LoRA found in: layer4.1.conv1
LoRA found in: layer4.1.conv2


In [18]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)

In [19]:

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total
    return avg_loss, acc


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total
    return avg_loss, acc

In [20]:
num_epochs = 8
best_val_acc = 0.0

trainable_params = count_trainable_params(model)
print(f"Trainable params: {trainable_params:,}")

for epoch in range(num_epochs):

    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    epoch_time = time.time() - start_time
    peak_memory = get_peak_memory()

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")
    print(f"Time: {epoch_time:.2f}s | Peak Mem: {peak_memory:.2f} MB")
    print("-" * 40)

    # SAVE METRICS TO CSV
    with open(csv_file, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            'resnet_real_Lora',
            epoch + 1,
            train_loss,
            train_acc,
            val_loss,
            val_acc,
            epoch_time,
            peak_memory,
            trainable_params
        ])

    # SAVE BEST MODEL
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save({
            "base_weights": "resnet18_imagenet1k_v1",  # IMPORTANT: store identity, not object
            "model_state_dict": model.state_dict(),     # FULL model (safest fallback)
            "lora_state_dict": lora.lora_state_dict(model),
            "fc_state_dict": model.fc.state_dict(),
            "epoch": epoch,
            "val_acc": val_acc
        }, "models/lora_full_checkpoint_april_21.pth")

        print("Best LoRA checkpoint saved!")

Trainable params: 572,517
Epoch 1/8
Train Loss: 2.1672, Train Acc: 0.4587
Val Loss:   1.9174, Val Acc:   0.5208
Time: 235.98s | Peak Mem: 643.83 MB
----------------------------------------
Best LoRA checkpoint saved!
Epoch 2/8
Train Loss: 1.7285, Train Acc: 0.5559
Val Loss:   1.8262, Val Acc:   0.5479
Time: 165.53s | Peak Mem: 643.83 MB
----------------------------------------
Best LoRA checkpoint saved!
Epoch 3/8
Train Loss: 1.6176, Train Acc: 0.5809
Val Loss:   1.8428, Val Acc:   0.5508
Time: 160.02s | Peak Mem: 643.83 MB
----------------------------------------
Best LoRA checkpoint saved!
Epoch 4/8
Train Loss: 1.5450, Train Acc: 0.5967
Val Loss:   1.7866, Val Acc:   0.5546
Time: 164.23s | Peak Mem: 643.83 MB
----------------------------------------
Best LoRA checkpoint saved!
Epoch 5/8
Train Loss: 1.4828, Train Acc: 0.6112
Val Loss:   1.7254, Val Acc:   0.5770
Time: 157.78s | Peak Mem: 643.83 MB
----------------------------------------
Best LoRA checkpoint saved!
Epoch 6/8
Train Los